# EmpathBot_V1 — Final Training Notebook
**CS731 / COMPSYS 731 | Group 14**

## Pipeline
```
1. Merged dataset loaded directly from Kaggle input (no rebuild needed)
2. Pre-trained ResNet-18 checkpoint → keys remapped → EmpathBotV1 backbone init
3. Fine-tune EmpathBotV1 on merged dataset (6 EmpathBot classes)
4. Evaluate → per-class recall, confusion matrix, comparison table
```

## Two inputs required in Kaggle
| Kaggle dataset | Path | Contents |
|---|---|---|
| `empathbot-merged-dataset` | `/kaggle/input/empathbot-merged-dataset` | AffectNet + RAF-DB + SFEW + FER2013, layout `{split}/{eb_label}/` |
| Your ResNet-18 checkpoint | `/kaggle/input/your-resnet/resnet18.pt` | Plain `resnet18` `state_dict` |

## EmpathBot 6-class label system
| eb_label | Name | Source emotions |
|---|---|---|
| 0 | neutral | AffectNet Neutral |
| 1 | trust_relief | AffectNet Happy |
| 2 | sadness | AffectNet Sad |
| 3 | fear_anxiety | AffectNet Fear |
| 4 | confusion | AffectNet Surprise |
| 5 | distrust | AffectNet Disgust + Anger + Contempt |

**GPU: T4 x2 or P100 — Kaggle Settings → Accelerator**


## 0. Environment

In [ ]:
import json, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, recall_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


## 1. Paths & Labels

Update `MERGED_DIR` if your dataset slug differs.  
The ResNet-18 path is auto-discovered in **Section 1.5** — no manual path needed.


In [ ]:
# ── Original processed dataset (dataset-facial on Kaggle) ────────────────────
BASE_DIR   = Path('/kaggle/input/dataset-facial')        # <-- UPDATE if slug differs
MASTER_CSV = BASE_DIR / 'data' / 'details' / 'master_split.csv'

# ── FER2013 CSV dataset ───────────────────────────────────────────────────────
FER2013_DIR = Path('/kaggle/input/fer2013')               # <-- UPDATE if slug differs

# ── Merged dataset (set after first upload; checked automatically below) ─────
MERGED_KAGGLE_SLUG = 'empathbot-merged-dataset'           # <-- your dataset slug
KAGGLE_USERNAME    = 'prenz1'                             # <-- your Kaggle username
MERGED_KAGGLE_DIR  = Path(f'/kaggle/input/{MERGED_KAGGLE_SLUG}')

# Working dir where merged data is built (Kaggle /kaggle/working is writable)
MERGED_BUILD_DIR   = Path('/kaggle/working/merged_dataset')

# MERGED_DIR points to already-uploaded Kaggle input if it exists,
# otherwise to the local build dir we create in Section 2.
MERGED_DIR = MERGED_KAGGLE_DIR if MERGED_KAGGLE_DIR.exists() else MERGED_BUILD_DIR

# ── Pre-trained ResNet-18 (Kaggle Model: preetiv1/trained-resnet-18) ────────
RESNET_MODEL_DIR = Path('/kaggle/input/trained-resnet-18')
RESNET_CKPT      = None   # set automatically in Section 1.5

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('/kaggle/working/empathbot_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUTPUT_DIR / 'empathbot_v1_best.pt'

# ── EmpathBot 6-class label system ────────────────────────────────────────────
EMPATHBOT_CLASSES = {0:'neutral', 1:'trust_relief', 2:'sadness',
                     3:'fear_anxiety', 4:'confusion', 5:'distrust'}
NUM_CLASSES      = len(EMPATHBOT_CLASSES)
CLASSES          = [EMPATHBOT_CLASSES[i] for i in range(NUM_CLASSES)]
CLASS_TO_IDX     = {v: k for k, v in EMPATHBOT_CLASSES.items()}
NEGATIVE_CLASSES = ['sadness', 'fear_anxiety', 'confusion', 'distrust']

# FER2013 label → EmpathBot label
FER_TO_EB = {0:5, 1:5, 2:3, 3:1, 4:2, 5:4, 6:0}

# ── Baseline recall (from notebook 2_benchmark_resnet18) ─────────────────────
BASELINE_RECALL = {
    'neutral':0.980, 'trust_relief':0.934, 'distrust':0.773,
    'confusion':0.642, 'fear_anxiety':0.596, 'sadness':0.556,
}

if MERGED_KAGGLE_DIR.exists():
    print(f'✅ Merged dataset already on Kaggle → {MERGED_KAGGLE_DIR}')
    print('   Skip Sections 2–2.4 and go straight to Section 1.5 (ResNet discovery).')
else:
    print('⚠️  Merged dataset not found on Kaggle — need to rebuild.')
    print(f'   Will build to {MERGED_BUILD_DIR} then upload (Section 2.4).')
    print(f'   Make sure these inputs are added to this notebook:')
    print(f'     • dataset-facial      → {BASE_DIR}  (exists={BASE_DIR.exists()})')
    print(f'     • fer2013             → {FER2013_DIR}  (exists={FER2013_DIR.exists()})')

print(f'\nMERGED_DIR = {MERGED_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')


## 1.5. Auto-discover ResNet-18 Checkpoint

Kaggle Models mount at `/kaggle/input/{slug}/{framework}/{variation}/{version}/`.  
This cell walks the model directory, finds every `.pt` / `.pth` file, and sets `RESNET_CKPT`
automatically — no need to know the exact framework/variation/version path.


In [ ]:
# ── Walk the model directory and find all checkpoint files ────────────────────
if not RESNET_MODEL_DIR.exists():
    raise FileNotFoundError(
        f'Model directory not found: {RESNET_MODEL_DIR}\n'
        f'Make sure you added "preetiv1/trained-resnet-18" as an input '
        f'to this notebook (Notebook Settings → Add data → Models).'
    )

# Show full directory tree
print(f'Contents of {RESNET_MODEL_DIR}:')
for p in sorted(RESNET_MODEL_DIR.rglob('*')):
    indent = '  ' * (len(p.relative_to(RESNET_MODEL_DIR).parts) - 1)
    size   = f'  ({p.stat().st_size/1e6:.1f} MB)' if p.is_file() else '/'
    print(f'  {indent}{p.name}{size}')

# Auto-select the checkpoint file
ckpt_files = sorted(RESNET_MODEL_DIR.rglob('*.pt')) + \
             sorted(RESNET_MODEL_DIR.rglob('*.pth'))

if len(ckpt_files) == 0:
    raise FileNotFoundError(
        f'No .pt or .pth files found under {RESNET_MODEL_DIR}. '
        f'Check the model was uploaded correctly on Kaggle.'
    )

if len(ckpt_files) == 1:
    RESNET_CKPT = ckpt_files[0]
else:
    # Multiple files — pick the largest (most likely the full model, not a config)
    RESNET_CKPT = max(ckpt_files, key=lambda p: p.stat().st_size)
    print(f'\n⚠️  Multiple checkpoint files found — using largest:')
    for f in ckpt_files:
        marker = ' ← selected' if f == RESNET_CKPT else ''
        print(f'  {f}  ({f.stat().st_size/1e6:.1f} MB){marker}')

print(f'\n✅ RESNET_CKPT = {RESNET_CKPT}')

# Peek at the checkpoint to confirm format before Section 5
import torch
raw = torch.load(RESNET_CKPT, map_location='cpu', weights_only=False)
if isinstance(raw, dict):
    keys = list(raw.keys())
    print(f'Checkpoint is a dict with keys: {keys}')
    # Show sample weight keys from whichever sub-key holds the state dict
    sd = raw.get('model_state_dict', raw)
    sample_keys = list(sd.keys())[:5]
else:
    # Raw state dict
    sample_keys = list(raw.keys())[:5]
print(f'Sample weight keys: {sample_keys}')
del raw   # free memory — we reload properly in Section 5


## 2. Rebuild Merged Dataset

**Only run this section if the merged dataset is not on Kaggle.**  
If `MERGED_KAGGLE_DIR.exists()` printed ✅ above, skip to **Section 1.5**.

Steps:
1. **2.1** — Inspect the original dataset via `master_split.csv`
2. **2.2** — Extract underperforming classes from FER2013 CSV
3. **2.3** — Copy everything into `MERGED_BUILD_DIR/{split}/{eb_label}/`
4. **2.4** — Upload `MERGED_BUILD_DIR` to Kaggle as `empathbot-merged-dataset`


In [ ]:
import shutil
from tqdm.notebook import tqdm

# ── 2.1  Inspect original dataset ────────────────────────────────────────────
df_master = pd.read_csv(MASTER_CSV)
df_master['split']      = df_master['split'].replace({'eval': 'val'})
df_master['class_name'] = df_master['eb_label'].map(EMPATHBOT_CLASSES)

print(f'master_split.csv — {len(df_master):,} rows')
print(df_master.groupby(['dataset','split']).size().unstack(fill_value=0))

train_counts = (
    df_master[df_master['split']=='train']
    .groupby('class_name').size()
    .reindex(CLASSES, fill_value=0)
)
print('\nTrain counts per class:')
print(train_counts.to_string())

colors = ['#d32f2f' if c in NEGATIVE_CLASSES else '#1976d2' for c in CLASSES]
fig, ax = plt.subplots(figsize=(11,4))
bars = ax.bar(train_counts.index, train_counts.values, color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Training samples BEFORE FER2013 top-up (red = hard classes)')
ax.set_ylabel('Images')
ax.axhline(train_counts.mean(), color='gray', ls='--', label=f'Mean {train_counts.mean():.0f}')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_dist_before.png', dpi=150)
plt.show()


In [ ]:
# ── 2.2  Extract FER2013 → sadness / fear_anxiety / confusion only ───────────
FER_IMPORT_EB  = {2, 3, 4}   # sadness, fear_anxiety, confusion
FER_CAP        = 2000        # max images per class from FER2013
FER_STAGE_DIR  = Path('/kaggle/working/fer2013_staged')

def decode_fer(pixel_str: str) -> 'np.ndarray':
    return np.array(pixel_str.split(), dtype=np.uint8).reshape(48, 48)

csv_path = next(FER2013_DIR.rglob('fer2013.csv'), None)
if csv_path is None:
    raise FileNotFoundError(f'fer2013.csv not found under {FER2013_DIR}')

df_fer = pd.read_csv(csv_path)
split_map = {'Training':'train','PublicTest':'val','PrivateTest':'test'}
counters = Counter()
fer_records = []

for _, row in tqdm(df_fer.iterrows(), total=len(df_fer), desc='Decoding FER2013'):
    eb = FER_TO_EB[int(row['emotion'])]
    if eb not in FER_IMPORT_EB or counters[eb] >= FER_CAP:
        continue
    split   = split_map.get(row['Usage'], 'train')
    out_dir = FER_STAGE_DIR / split / str(eb)
    out_dir.mkdir(parents=True, exist_ok=True)
    img = Image.fromarray(decode_fer(str(row['pixels'])), mode='L').convert('RGB')
    img = img.resize((224, 224), Image.LANCZOS)
    fname = f'fer_{counters[eb]:05d}.png'
    img.save(out_dir / fname)
    counters[eb] += 1
    fer_records.append({'split':split,'eb_label':eb,
                        'class_name':EMPATHBOT_CLASSES[eb],'source':'fer'})

print('FER2013 extracted:')
for eb, n in sorted(counters.items()):
    print(f'  eb_label {eb} ({EMPATHBOT_CLASSES[eb]:15s}): {n:,}')


In [ ]:
# ── 2.3  Merge original dataset + FER2013 into MERGED_BUILD_DIR ──────────────
MERGED_BUILD_DIR.mkdir(parents=True, exist_ok=True)

# Copy original images (paths in master_split.csv are relative to BASE_DIR)
orig_records = []
for row in tqdm(df_master.itertuples(), total=len(df_master), desc='Copying original'):
    src = BASE_DIR / row.path
    if not src.exists():
        continue
    dst_dir = MERGED_BUILD_DIR / row.split / str(row.eb_label)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f'orig_{src.name}'
    if not dst.exists():
        shutil.copy2(src, dst)
    orig_records.append({'split':row.split,'eb_label':row.eb_label,
                         'class_name':EMPATHBOT_CLASSES[row.eb_label],'source':row.dataset})

# Copy FER2013 staged images
fer_records_copy = []
for split_dir in sorted(FER_STAGE_DIR.iterdir()):
    for eb_dir in sorted(split_dir.iterdir()):
        eb = int(eb_dir.name)
        dst_dir = MERGED_BUILD_DIR / split_dir.name / str(eb)
        dst_dir.mkdir(parents=True, exist_ok=True)
        for img in eb_dir.glob('*.png'):
            dst = dst_dir / img.name
            if not dst.exists():
                shutil.copy2(img, dst)
            fer_records_copy.append({'split':split_dir.name,'eb_label':eb,
                                     'class_name':EMPATHBOT_CLASSES[eb],'source':'fer'})

df_merged = pd.concat([
    pd.DataFrame(orig_records),
    pd.DataFrame(fer_records_copy)
], ignore_index=True)

summary = df_merged.groupby(['split','class_name']).size().unstack(fill_value=0)
print('=== Merged Dataset Summary ===')
print(summary.reindex(CLASSES))
print(f'\nTotal: {len(df_merged):,} images')
print(f'Built at: {MERGED_BUILD_DIR}')

# Save manifest for provenance
df_merged.to_csv(MERGED_BUILD_DIR / 'merged_manifest.csv', index=False)

# Update MERGED_DIR to point to the freshly built directory
MERGED_DIR = MERGED_BUILD_DIR
print(f'MERGED_DIR updated → {MERGED_DIR}')


### 2.4. Upload Merged Dataset to Kaggle

Run once after the merge completes.  
Set `CREATE_NEW = False` on subsequent runs to add a new version instead.

After upload (~5 min processing):
1. Add `prenz1/empathbot-merged-dataset` as an **input dataset** to this notebook
2. `MERGED_KAGGLE_DIR` will exist on the next run → Sections 2–2.4 are skipped automatically


In [ ]:
import subprocess

CREATE_NEW = True   # False after first upload — adds a new version instead

meta = {
    'title': 'EmpathBot Merged Dataset',
    'id': f'{KAGGLE_USERNAME}/{MERGED_KAGGLE_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
    'description': (
        'AffectNet-HQ + RAF-DB + SFEW + FER2013 top-up (sadness/fear_anxiety/confusion). '
        'Layout: {split}/{eb_label}/*.{jpg,png}. '
        'EmpathBot 6-class: 0=neutral 1=trust_relief 2=sadness '
        '3=fear_anxiety 4=confusion 5=distrust.'
    ),
}
with open(MERGED_BUILD_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

n_imgs = sum(1 for p in MERGED_BUILD_DIR.rglob('*')
             if p.suffix.lower() in {'.jpg','.jpeg','.png'})
print(f'Uploading {n_imgs:,} images from {MERGED_BUILD_DIR} ...')

cmd = (['kaggle','datasets','create','-p', str(MERGED_BUILD_DIR)] if CREATE_NEW
       else ['kaggle','datasets','version','-p', str(MERGED_BUILD_DIR),
              '-m','Rebuild: affectnet+rafdb+sfew+fer2013'])

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Upload failed — see stderr above')

print('✅ Upload complete!')
print(f'   https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{MERGED_KAGGLE_SLUG}')
print()
print('Next steps:')
print('  1. Wait ~5 min for Kaggle to process the dataset.')
print(f'  2. Add it as an input: Notebook Settings → Add data → {MERGED_KAGGLE_SLUG}')
print('  3. Re-run from Section 1 — Sections 2-2.4 will be skipped automatically.')


## 3. Dataset & DataLoaders

Reads images directly from the Kaggle input — no file copying.  
Negative/hard emotion classes get stronger augmentation (larger crop, more jitter,
random grayscale) to improve their per-class recall.


In [ ]:
class EmpathBotDataset(Dataset):
    """
    Folder layout: {root}/{split}/{eb_label}/*.{jpg,jpeg,png}
    eb_label is an integer 0-5 stored as the folder name.
    Negative/hard classes receive stronger augmentation during training.
    """
    _NEGATIVE_IDX = {CLASS_TO_IDX[c] for c in NEGATIVE_CLASSES}
    _MEAN = [0.485, 0.456, 0.406]
    _STD  = [0.229, 0.224, 0.225]

    def __init__(self, root: Path, split: str):
        self.is_train = (split == 'train')
        self.samples  = []
        for eb_dir in sorted((root / split).iterdir(), key=lambda p: int(p.name)):
            label = int(eb_dir.name)
            if label not in EMPATHBOT_CLASSES:
                continue
            for f in eb_dir.glob('*'):
                if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                    self.samples.append((f, label))

        self._std_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(self._MEAN, self._STD),
        ])
        self._neg_tf = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            transforms.ToTensor(),
            transforms.Normalize(self._MEAN, self._STD),
        ])
        self._val_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(self._MEAN, self._STD),
        ])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.is_train:
            tf = self._neg_tf if label in self._NEGATIVE_IDX else self._std_tf
        else:
            tf = self._val_tf
        return tf(img), label

    def class_counts(self) -> list:
        c = Counter(lbl for _, lbl in self.samples)
        return [c.get(i, 0) for i in range(NUM_CLASSES)]


BATCH_SIZE  = 64
NUM_WORKERS = 2

train_ds = EmpathBotDataset(MERGED_DIR, 'train')
val_ds   = EmpathBotDataset(MERGED_DIR, 'val')
test_ds  = EmpathBotDataset(MERGED_DIR, 'test')

# No WeightedRandomSampler — class-weighted loss handles imbalance;
# using both causes double-penalisation and training oscillation.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
print('\nClass counts (train):')
for i, n in enumerate(train_ds.class_counts()):
    mark = ' ← hard' if EMPATHBOT_CLASSES[i] in NEGATIVE_CLASSES else ''
    print(f'  {EMPATHBOT_CLASSES[i]:15s} [{i}]: {n:,}{mark}')


## 4. EmpathBotV1 Architecture

ResNet-18 backbone (all conv/BN/residual blocks) + SE channel-attention + 3-layer head.

| Component | Shape | Notes |
|---|---|---|
| Backbone | `(B,512,7,7)` | ResNet-18 layers 0–7, weights loaded from checkpoint |
| SE attention | `(B,512,7,7)` | Focuses on discriminative face channels |
| GAP | `(B,512)` | Global average pool |
| Head | `512→256→128→6` | BatchNorm + Dropout(0.4/0.2) after each hidden layer |

**Key mapping (plain ResNet-18 → EmpathBotV1):**
```
conv1.*  → backbone.0.*     bn1.*    → backbone.1.*
layer1.* → backbone.4.*     layer2.* → backbone.5.*
layer3.* → backbone.6.*     layer4.* → backbone.7.*
fc.*     → skipped  (replaced by custom 3-layer head)
```


In [ ]:
class SqueezeExcitation(nn.Module):
    """Lightweight channel attention block (~4k extra params)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)


class EmpathBotV1(nn.Module):
    """
    6-class emotion model for healthcare/HRI settings.
    Backbone = ResNet-18 wrapped in nn.Sequential (strips avgpool+fc).
    Backbone weights are loaded from a pre-trained ResNet-18 checkpoint;
    SE attention and custom head start from random init.
    """
    def __init__(self, num_classes: int = 6, dropout: float = 0.4):
        super().__init__()
        bb = models.resnet18(weights=None)   # weights loaded separately via load_resnet()
        self.backbone   = nn.Sequential(*list(bb.children())[:-2])  # → (B,512,7,7)
        self.attention  = SqueezeExcitation(512, reduction=16)
        self.gap        = nn.AdaptiveAvgPool2d(1)                   # → (B,512)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout*0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.attention(self.backbone(x))))

    def n_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


model = EmpathBotV1(num_classes=NUM_CLASSES).to(DEVICE)
print(f'EmpathBotV1 parameters: {model.n_params():,}')
with torch.no_grad():
    out = model(torch.randn(4, 3, 224, 224).to(DEVICE))
    print(f'Forward pass OK: {out.shape}  ✅')


## 5. Load Pre-trained ResNet-18 → Initialise Backbone

The checkpoint uses standard PyTorch ResNet-18 key names.  
We remap them to EmpathBotV1's `backbone.*` keys, skip `fc.*`, then load with `strict=False`  
so the SE-attention and classifier layers keep their random init and will be trained from scratch.


In [ ]:
_REMAP = {
    'conv1.':  'backbone.0.',
    'bn1.':    'backbone.1.',
    'layer1.': 'backbone.4.',
    'layer2.': 'backbone.5.',
    'layer3.': 'backbone.6.',
    'layer4.': 'backbone.7.',
}

def load_resnet_backbone(model: EmpathBotV1, ckpt_path: Path) -> None:
    """
    Load a plain ResNet-18 checkpoint into EmpathBotV1.
    Supports:
      - raw state_dict  (keys start with conv1 / layer1 / ...)
      - wrapped dict    {'model_state_dict': state_dict, ...}
      - EmpathBotV1 checkpoint  (keys start with backbone.)
    """
    raw = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if isinstance(raw, dict) and 'model_state_dict' in raw:
        src = raw['model_state_dict']
        print(f'Checkpoint metadata: epoch={raw.get("epoch","?")}, '
              f'val_acc={raw.get("val_acc","?")}')
    else:
        src = raw

    sample = next(iter(src))
    print(f'First key: "{sample}"')

    if sample.startswith('backbone.'):
        # Already an EmpathBotV1 checkpoint — load directly
        model.load_state_dict(src, strict=True)
        print('Loaded EmpathBotV1 checkpoint directly ✅')
        return

    if not sample.startswith(('conv1.', 'bn1.', 'layer')):
        raise ValueError(f'Unrecognised checkpoint format. Sample key: "{sample}"')

    # Remap plain ResNet-18 keys → EmpathBotV1 backbone keys
    remapped, skipped = {}, []
    for k, v in src.items():
        if k.startswith('fc.'):
            skipped.append(k)
            continue
        new_k = k
        for prefix, repl in _REMAP.items():
            if k.startswith(prefix):
                new_k = repl + k[len(prefix):]
                break
        remapped[new_k] = v

    missing, unexpected = model.load_state_dict(remapped, strict=False)

    backbone_missing = [k for k in missing if k.startswith('backbone.')]
    fresh_init       = [k for k in missing if not k.startswith('backbone.')]

    print(f'\nKey remapping summary:')
    print(f'  Backbone keys loaded : {len(remapped)}')
    print(f'  Skipped (fc.*)       : {skipped}')
    if backbone_missing:
        print(f'  ⚠️  Missing backbone  : {backbone_missing}')
    else:
        print(f'  Backbone fully loaded ✅')
    print(f'  Fresh init (expected): {len(fresh_init)} tensors  '
          f'← attention + classifier')


load_resnet_backbone(model, RESNET_CKPT)


## 6. Loss — Class-Weighted Cross-Entropy

Inverse-frequency weights with a mild extra boost (×1.2) on negative classes.  
Label smoothing = 0.05 (not 0.1 — with 6 classes and MixUp, heavier smoothing made the model too underconfident).


In [ ]:
def compute_class_weights(ds: EmpathBotDataset, neg_boost: float = 1.2) -> torch.Tensor:
    counts = ds.class_counts()
    total  = sum(counts)
    w = [total / (NUM_CLASSES * max(c, 1)) for c in counts]
    neg_idx = {CLASS_TO_IDX[c] for c in NEGATIVE_CLASSES}
    for i in neg_idx:
        w[i] *= neg_boost
    wt = torch.tensor(w, dtype=torch.float32)
    wt = wt / wt.sum() * NUM_CLASSES   # normalise: mean ≈ 1

    print('Class weights (higher = penalised more for wrong predictions):')
    for i, name in EMPATHBOT_CLASSES.items():
        mark = ' ← boosted' if i in neg_idx else ''
        print(f'  {name:15s} [{i}]: {wt[i]:.3f}{mark}')
    return wt


class_weights = compute_class_weights(train_ds, neg_boost=1.2).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
print('\nLoss: CrossEntropyLoss(class_weights, label_smoothing=0.05)')


## 7. Optimiser & Scheduler

**Two-group LR:** backbone (pre-trained) gets 10× lower LR than the fresh SE + head layers.  
**Schedule:** 3-epoch linear warm-up → cosine annealing to zero.  
**Delayed MixUp:** disabled for the first 10 epochs so the model first learns class boundaries,
then activated at α=0.2 as regularisation for the remaining epochs.


In [ ]:
LR_HEAD           = 3e-3   # SE-attention + classifier — fresh weights, need fast convergence
LR_BACKBONE       = 1e-4   # pre-trained layers — fine-tune slowly
WEIGHT_DECAY      = 1e-4
EPOCHS            = 40
MIXUP_START_EPOCH = 10     # MixUp activates after this epoch
MIXUP_ALPHA       = 0.2

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(),   'lr': LR_BACKBONE},
    {'params': model.attention.parameters(),  'lr': LR_HEAD},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

def _lr_lambda(epoch):
    if epoch < 3:
        return (epoch + 1) / 3
    return 0.5 * (1 + np.cos(np.pi * (epoch - 3) / (EPOCHS - 3)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

print(f'Backbone LR : {LR_BACKBONE}   Head LR: {LR_HEAD}   WD: {WEIGHT_DECAY}')
print(f'Epochs      : {EPOCHS}   MixUp from epoch {MIXUP_START_EPOCH + 1} (alpha={MIXUP_ALPHA})')
print(f'Schedule    : 3-epoch warmup → cosine annealing')


## 8. Training

In [ ]:
def mixup_batch(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(crit, pred, y_a, y_b, lam):
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)


def train_epoch(model, loader, optimizer, criterion, device, use_mixup: bool):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if use_mixup:
            imgs, y_a, y_b, lam = mixup_batch(imgs, labels, MIXUP_ALPHA)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = mixup_loss(criterion, out, y_a, y_b, lam) if use_mixup else criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = out.argmax(1)
        # Measure against dominant label (y_a) so metric is readable even with MixUp on
        ref = y_a if use_mixup else labels
        correct += preds.eq(ref).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item()
        preds = out.argmax(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100.0 * correct / total, all_preds, all_labels


print('Training functions ready.')


In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = best_epoch = 0

hdr = f"{'Ep':>3} {'Mix':>3} {'TrLoss':>8} {'TrAcc':>7} {'VlLoss':>8} {'VlAcc':>7} {'HeadLR':>9}"
print(hdr)
print('-' * len(hdr))

for epoch in range(1, EPOCHS + 1):
    use_mixup = epoch > MIXUP_START_EPOCH

    tr_loss, tr_acc          = train_epoch(model, train_loader, optimizer, criterion, DEVICE, use_mixup)
    vl_loss, vl_acc, _, _    = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    mark = ''
    if vl_acc > best_val_acc:
        best_val_acc, best_epoch = vl_acc, epoch
        torch.save({
            'epoch': epoch, 'val_acc': vl_acc,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'empathbot_classes': EMPATHBOT_CLASSES,
            'architecture': 'EmpathBotV1',
        }, CHECKPOINT)
        mark = ' ✓'

    head_lr = optimizer.param_groups[1]['lr']
    mx = 'on' if use_mixup else 'off'
    print(f"{epoch:>3} {mx:>3} {tr_loss:>8.4f} {tr_acc:>6.2f}% "
          f"{vl_loss:>8.4f} {vl_acc:>6.2f}%{mark} {head_lr:.2e}")

print(f'\nBest val acc: {best_val_acc:.2f}% at epoch {best_epoch}')
print(f'Checkpoint saved → {CHECKPOINT}')


## 9. Training Curves

In [ ]:
epochs_x = range(1, len(history['train_acc']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, history['train_acc'], label='Train', color='#1976d2')
axes[0].plot(epochs_x, history['val_acc'],   label='Val',   color='#d32f2f')
axes[0].axvline(best_epoch, color='gray', ls='--', alpha=0.6, label=f'Best ({best_epoch})')
axes[0].axvline(MIXUP_START_EPOCH + 0.5, color='orange', ls=':', alpha=0.8, label='MixUp on')
axes[0].set_title('EmpathBot_V1 — Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['train_loss'], label='Train', color='#1976d2')
axes[1].plot(epochs_x, history['val_loss'],   label='Val',   color='#d32f2f')
axes[1].axvline(best_epoch, color='gray', ls='--', alpha=0.6)
axes[1].axvline(MIXUP_START_EPOCH + 0.5, color='orange', ls=':', alpha=0.8)
axes[1].set_title('EmpathBot_V1 — Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'Best Val Acc: {best_val_acc:.2f}% @ epoch {best_epoch}', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()


## 10. Test Set Evaluation

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Best checkpoint: epoch {ckpt['epoch']} | val acc {ckpt['val_acc']:.2f}%")

_, test_acc, test_preds, test_labels = eval_epoch(model, test_loader, criterion, DEVICE)
print(f'Test accuracy: {test_acc:.2f}%\n')
print(classification_report(test_labels, test_preds, target_names=CLASSES))


In [ ]:
new_recall = recall_score(test_labels, test_preds,
                          labels=list(range(NUM_CLASSES)), average=None)

print(f"{'Class':<17} {'Baseline':>10} {'EmpathBotV1':>12} {'Delta':>8}")
print('-' * 52)
for i, name in EMPATHBOT_CLASSES.items():
    old = BASELINE_RECALL.get(name)
    new = new_recall[i]
    if old is not None:
        d = new - old
        arrow = '↑' if d >= 0 else '↓'
        print(f"{name:<17} {old*100:>9.1f}%  {new*100:>11.1f}%  {arrow}{abs(d*100):>5.1f}%")
    else:
        print(f"{name:<17} {'—':>10}  {new*100:>11.1f}%")


## 11. Confusion Matrix

In [ ]:
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',            'd'),
    (axes[1], cm_norm, 'Recall-normalised', '.2f'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('EmpathBot_V1 — Test Set', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()


## 12. Per-class Recall Bar Chart (Report-ready)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x, w = np.arange(NUM_CLASSES), 0.35

b_vals = [BASELINE_RECALL.get(EMPATHBOT_CLASSES[i], 0) for i in range(NUM_CLASSES)]
n_vals = list(new_recall)

b1 = ax.bar(x - w/2, [v*100 for v in b_vals], w,
            label='Baseline ResNet-18', color='#90a4ae', edgecolor='white')
b2 = ax.bar(x + w/2, [v*100 for v in n_vals], w,
            label='EmpathBot_V1', color='#0f4c5c', edgecolor='white')
ax.bar_label(b1, fmt='%.0f%%', padding=3, fontsize=8, color='#555')
ax.bar_label(b2, fmt='%.0f%%', padding=3, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Recall (%)')
ax.set_ylim(0, 115)
ax.axhline(70, color='#d32f2f', ls='--', alpha=0.6, label='Target 70%')
ax.set_title('Per-class Recall: Baseline ResNet-18 vs EmpathBot_V1',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, name in enumerate(CLASSES):
    if name in NEGATIVE_CLASSES:
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.06, color='#d32f2f', zorder=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'recall_comparison.png', dpi=150)
plt.show()


## 13. Architecture Comparison Table (Report Section 5)

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'ResNet-18 (baseline)',
     'Dataset': 'AffectNet-HQ + RAF-DB + SFEW',
     'Val Acc': 'YOUR_BASELINE',
     'Sadness Recall': f"{BASELINE_RECALL['sadness']*100:.1f}%",
     'Fear Recall':    f"{BASELINE_RECALL['fear_anxiety']*100:.1f}%",
     'Params': '11.2 M',
     'Notes': 'Standard FC head, no class weighting'},
    {'Model': 'EmpathBot_V1',
     'Dataset': 'AffectNet-HQ + RAF-DB + SFEW + FER2013',
     'Val Acc': f'{best_val_acc:.2f}%',
     'Sadness Recall': f"{new_recall[CLASS_TO_IDX['sadness']]*100:.1f}%",
     'Fear Recall':    f"{new_recall[CLASS_TO_IDX['fear_anxiety']]*100:.1f}%",
     'Params': f'{model.n_params()/1e6:.1f} M',
     'Notes': 'SE-attention + 3-layer head + class weights + delayed MixUp (α=0.2)'},
    {'Model': 'POSTER++ (literature)',
     'Dataset': 'RAF-DB', 'Val Acc': '92.21%',
     'Sadness Recall': '—', 'Fear Recall': '—', 'Params': '~43 M',
     'Notes': 'Mao et al. 2023'},
    {'Model': 'Ada-DF (literature)',
     'Dataset': 'RAF-DB + AffectNet', 'Val Acc': '~90%',
     'Sadness Recall': '—', 'Fear Recall': '—', 'Params': '—',
     'Notes': 'Liu et al. 2024'},
])
print(comparison.to_string(index=False))
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
print('\nSaved model_comparison.csv')


## 14. Save Outputs

In [ ]:
with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

with open(OUTPUT_DIR / 'class_mapping.json', 'w') as f:
    json.dump({'empathbot_classes': EMPATHBOT_CLASSES,
               'class_to_idx': CLASS_TO_IDX,
               'idx_to_class': {str(v): k for k, v in CLASS_TO_IDX.items()}}, f, indent=2)

print(f'Outputs in {OUTPUT_DIR}:')
for p in sorted(OUTPUT_DIR.iterdir()):
    sz = p.stat().st_size
    unit, val = ('MB', sz/1e6) if sz > 1e6 else ('KB', sz/1e3)
    print(f'  {p.name:<42} {val:6.1f} {unit}')

print(f'\n{"="*58}')
print('EmpathBot_V1 — Final Results')
print(f'  Best Val Accuracy  : {best_val_acc:.2f}%  (epoch {best_epoch})')
for name in ['sadness', 'fear_anxiety', 'confusion']:
    i   = CLASS_TO_IDX[name]
    old = BASELINE_RECALL[name]
    new = new_recall[i]
    arrow = '↑' if new >= old else '↓'
    print(f'  {name:<18}: {new*100:.1f}%  {arrow}  (was {old*100:.1f}%)')
print(f'{"="*58}')
print('Submit: empathbot_v1_best.pt + class_mapping.json')
